# Anima — RTX PRO 6000 TRAINER (thin shell over the repo runner)

The training half of the pipeline: PULLS the cache the Colab A100 factory built + pushed to HF,
reconstructs the images byte-identically, builds the long-run `before_after` config, and launches the
two-phase train **DETACHED** (so a kernel restart can't kill an ~18 h/epoch job). The **animetimm**
adapter is the final LoRA.

**Same thin-shell contract as the cache factory:** all logic is in the repo
(`geolip_anima_trainer.trainer_runner.TrainerRunner`); this notebook is just `clone → install → a few
`t.<step>()` calls`. **To change anything, edit the repo and `git pull` — the notebook stays as-is.**

**Two things before you start:**
1. **`HF_TOKEN`** in the env (`export HF_TOKEN=...`) — needed to pull the cache + back up checkpoints.
2. **`DATA_ROOT` must equal the factory's portable path** (default `/workspace/anima_data`). The latent
   fingerprint embeds absolute image paths, so a different path re-validates → wipes → re-caches. The
   factory symlinks `/workspace/anima_data`; here it's a real dir — keep the string identical.

> **License — NON-COMMERCIAL** (CircleStone NC + NVIDIA Open Model License; Cosmos derivative).

## 0 · Confirm the GPU + disk

Expect **RTX PRO 6000 Blackwell** (sm_120, ~96 GB). bf16 throughout; no fp8/flash-attn. `df -h` should
show enough room on the disk holding `DATA_ROOT` for the reconstructed images + cache (~106 GB full set).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h | grep -vE 'tmpfs|overlay|udev'

## 1 · Bootstrap — clone + install (→ restart ONCE the first time)

Clones this repo + diffusion-pipe and installs deps (the logic is in `anima_colab.install`, so a `git
pull` updates it with no notebook edit). On a persistent box this only really runs once; later sessions
skip it. **Run the cell; if the kernel restarts, run it again** (it then skips), then continue.

In [ ]:
import os, sys, subprocess
REPO = os.environ.get("ANIMA_REPO", "/workspace/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO):           # ensure_repo (git pull + diffusion-pipe) + pip; True the 1st time
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · (after restart) Run the trainer

Each cell is one `TrainerRunner` step. `setup()` pins `DATA_ROOT`/`HF_HOME`, authenticates HF, checks the
GPU (warns if not sm_120), and downloads the model. `prepare_dataset()` pulls the factory's cache and
rebuilds the images. `train()` launches **detached** — it returns immediately; watch with `t.tail()`.

Tunables are kwargs on the `TrainerRunner(...)` cell (e.g. `cache_repo=`, `backup_repo=`, `num_gpus=`,
`micro_batch=`, `epochs_vlm=`) or `ANIMA_*` env vars — no notebook logic to edit.

In [ ]:
from geolip_anima_trainer.trainer_runner import TrainerRunner
t = TrainerRunner()                 # DATA_ROOT=/workspace/anima_data (MUST match the factory); set kwargs to tune
t.setup()                           # env + HF auth + GPU check (sm_120) + model download

In [ ]:
t.prepare_dataset()                 # pull the factory-built cache from HF + reconstruct images; prune source parquet

In [ ]:
t.build_configs()                   # the long-run before_after training tomls (one lora toml per phase)

In [ ]:
t.train()                           # launch the two-phase train DETACHED (survives a kernel restart)

In [ ]:
t.tail()                            # watch the train log (run anytime); t.status() for disk + latest checkpoint

## Notes

- **One-shot:** `TrainerRunner().run_all()` does setup → prepare → build → train (detached).
- **Resume after an interruption** (checkpoints are on disk): `t.resume('vlm')` then `t.resume('animetimm')`
  — each relaunches detached with `--resume_from_checkpoint`.
- **Back up the LoRA** (optional; the persistent disk is the primary durability): set `backup_repo=` (or
  `ANIMA_BACKUP_REPO`) and run `t.backup()` to push the newest checkpoint to an HF *model* repo.
- **Get the LoRA:** the final adapter is the newest `DATA_ROOT/runs/animetimm/<timestamp>/epochN/` — ComfyUI
  safetensors + PEFT config; drop the safetensors into `ComfyUI/models/loras/`.
- **Reset-safe:** a fresh runtime just re-runs the cells — `setup()` re-derives everything, `prepare_dataset()`
  re-pulls/reuses the cache. The detached train keeps running across a *kernel* restart (not a box reboot).
- **Updating logic = `git pull`** (step 1 pulls every run); nothing here needs editing.